In [1]:
import pickle
with open('all_types_1000p_70000t.pkl', 'rb') as file:
    data = pickle.load(file)

data

,coords,trajectory_type
0,"[[-0.5989235263769103, -1.162521574332328, -2....",ATTM
1,"[[-0.08481573554460747, 2.319417467387251, 3.5...",ATTM
2,"[[0.4460439556191179, -0.005415868872246488, -...",ATTM
3,"[[0.005016187430309248, 0.6084933010787497, 0....",ATTM
4,"[[0.018019833626267964, 1.1039231638637528, 1....",ATTM
...,...,...
69995,"[[0.0, -0.7743617637528311, -0.546565095390306...",Brownian Motion
69996,"[[0.0, 1.7572467829976466, 2.0560173343255634,...",Brownian Motion
69997,"[[0.0, 1.17471633827359, 2.036867647381258, 2....",Brownian Motion
69998,"[[0.0, -0.38920630052789046, -0.84620013605425...",Brownian Motion


In [2]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class TrajectoryTransformer(nn.Module):
    def __init__(self, input_features=2, d_model=128, nhead=8, num_layers=3, 
                 dim_feedforward=512, num_classes=7, dropout=0.1):
        super().__init__()
        
        # Proyección inicial de características
        self.linear_in = nn.Linear(input_features, d_model)
        
        # Codificación posicional
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Capas Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Capas de clasificación
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        # x shape: (batch_size, 2, 1000)
        x = x.permute(0, 2, 1)  # (batch_size, 1000, 2)
        x = self.linear_in(x)    # (batch_size, 1000, d_model)
        x = self.pos_encoder(x)
        x = self.transformer(x)  # (batch_size, 1000, d_model)
        
        # Pooling temporal
        x = x.mean(dim=1)        # (batch_size, d_model)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import torch.optim as optim

# 1. Preprocesamiento de datos
class TrajectoryDataset(Dataset):
    def __init__(self, data, labels, transform=None):
        self.trajectories = data
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        trajectory = torch.tensor(self.trajectories[idx], dtype=torch.float32)
        label = self.labels[idx]
        if self.transform:
            trajectory = self.transform(trajectory)
        return trajectory, label

# Convertir las listas de coordenadas a arrays numpy
X = np.array([np.array([x, y]) for x, y in data.iloc[:, 0]])
y = data.iloc[:, 1].values

# Codificar las etiquetas
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split estratificado (70-15-15)
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Crear DataLoaders
batch_size = 256

train_dataset = TrajectoryDataset(X_train, y_train)
val_dataset = TrajectoryDataset(X_val, y_val)
test_dataset = TrajectoryDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=4, pin_memory=True)

# 2. Configuración del modelo y entrenamiento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TrajectoryTransformer(
    input_features=2,
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_feedforward=1024,
    num_classes=7,
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

# Programación de learning rate con warmup
def lr_lambda(epoch):
    warmup_epochs = 5
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    else:
        return 0.95 ** (epoch - warmup_epochs)

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_val_acc = 0.0
patience = 5
counter = 0

# Bucle de entrenamiento
for epoch in range(50):
    # Entrenamiento
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
    
    scheduler.step()
    
    # Validación
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/50")
    print(f"Train Loss: {train_loss/len(train_loader.dataset):.4f} | Val Loss: {val_loss/len(val_loader.dataset):.4f}")
    print(f"Val Accuracy: {val_acc:.2f}%")
    
    # Early stopping y guardar mejor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

# 3. Evaluación final con el mejor modelo
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

print("\nResultados finales en Test:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=label_encoder.classes_,
    digits=4
))